# ⚛️ Example 2: Hybrid Quantum VQE Molecular Screening (SMILES $\to \Delta G$) & Clinical Study Grant FinOps

This interactive Jupyter Notebook demonstrates how **`quantum-biomed-research-engine`** screens candidate drug ligands (`SMILES`) against target disease receptors (`KRAS-G12C`, `EGFR-T790M`, `BACE1`) using a parameterized Variational Quantum Eigensolver (**VQE**) ansatz (`RY/RZ` rotations + `CNOT` entanglement ladder) and tracks R&D compute spend per clinical study grant.

In [ ]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('..'))

from src.agents.quantum_molecular_screener import QuantumMolecularScreener
from src.finops.study_budget import ClinicalStudyBudgetController


## 1. Quantum VQE Molecular Docking & Ground-State Energy Simulation
We screen **Sotorasib (AMG-510 scaffold)** against the **KRAS-G12C** oncology receptor across 6 active-space qubits and 3 variational `RY/RZ + CNOT` layers.

In [ ]:
screener = QuantumMolecularScreener()
sotorasib_smiles = "C=CC(=O)N1CCN(CC1)C2=NC(=O)N(C=C2C3=C(C=CC=C3F)O)C4=C(C=CN=C4C(C)C)C"

result = screener.screen_molecule(
    molecule_smiles=sotorasib_smiles,
    target_receptor="KRAS-G12C",
    num_qubits=6,
    ansatz_layers=3,
)

print(f"Quantum Backend:         {result['quantum_backend_used']}")
print(f"VQE Ground Energy (E_0): {result['vqe_ground_state_energy_hartree']} Hartrees")
print(f"Binding Affinity (ΔG):   {result['binding_affinity_kcal_mol']} kcal/mol (Score: {result['binding_affinity_score']})")
print(f"Dominant Bitstring:      |{result['dominant_bitstring']}⟩ ({result['dominant_state_probability']*100:.1f}%)")
print(f"Verdict:                 {result['drug_likeness_verdict']}")
print("\n=== SYNTHESIZED QUANTUM ANSATZ CIRCUIT ===")
print(result['circuit_diagram_ascii'])

## 2. Pharmaceutical & Quantum R&D Clinical Study Grant FinOps
Each clinical study grant (`STUDY-ONCO-KRAS-2026`) is metered across both clinical NLP synthesis tokens and quantum VQE statevector volume (`num_qubits × circuit_depth`), enforcing hard HTTP 402 budget caps and comparing Sovereign LiteLLM + Spot L4 GPU NIM economics.

In [ ]:
finops = ClinicalStudyBudgetController(db_path="/tmp/notebook_finops.db")
finops.ensure_study_exists("STUDY-ONCO-KRAS-2026", allocated_budget_usd=2500.0)

tx_cost, remaining = finops.check_and_charge_quantum(
    clinical_study_id="STUDY-ONCO-KRAS-2026",
    num_qubits=result["num_qubits"],
    circuit_depth=result["circuit_depth"],
)

report = finops.get_study_report("STUDY-ONCO-KRAS-2026")
print(f"Quantum Screen Cost: ${tx_cost} USD | Remaining Study Grant: ${remaining} USD")
print(json.dumps(report.model_dump(), indent=2))